# 01 — Data Quality & Loading
Load the raw flotation plant CSV, clean it (decimals, dates, column names, duplicates), and load the result into MySQL as `plant_data`.

In [19]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

## Load & standardize
Raw file uses comma decimals (`decimal=","`, Brazilian format); parse dates and tidy column names.

In [3]:
df = pd.read_csv('MiningProcess_Flotation_Plant_Database.csv', decimal=",")

In [4]:
df.head(10)

,date,% Iron Feed,% Silica Feed,Starch Flow,Amina Flow,Ore Pulp Flow,Ore Pulp pH,Ore Pulp Density,Flotation Column 01 Air Flow,Flotation Column 02 Air Flow,...,Flotation Column 07 Air Flow,Flotation Column 01 Level,Flotation Column 02 Level,Flotation Column 03 Level,Flotation Column 04 Level,Flotation Column 05 Level,Flotation Column 06 Level,Flotation Column 07 Level,% Iron Concentrate,% Silica Concentrate
0,2017-03-10 01:00:00,55.2,16.98,3019.53,557.434,395.713,10.0664,1.74,249.214,253.235,...,250.884,457.396,432.962,424.954,443.558,502.255,446.370,523.344,66.91,1.31
1,2017-03-10 01:00:00,55.2,16.98,3024.41,563.965,397.383,10.0672,1.74,249.719,250.532,...,248.994,451.891,429.560,432.939,448.086,496.363,445.922,498.075,66.91,1.31
2,2017-03-10 01:00:00,55.2,16.98,3043.46,568.054,399.668,10.0680,1.74,249.741,247.874,...,248.071,451.240,468.927,434.610,449.688,484.411,447.826,458.567,66.91,1.31
3,2017-03-10 01:00:00,55.2,16.98,3047.36,568.665,397.939,10.0689,1.74,249.917,254.487,...,251.147,452.441,458.165,442.865,446.210,471.411,437.690,427.669,66.91,1.31
4,2017-03-10 01:00:00,55.2,16.98,3033.69,558.167,400.254,10.0697,1.74,250.203,252.136,...,248.928,452.441,452.900,450.523,453.670,462.598,443.682,425.679,66.91,1.31
5,2017-03-10 01:00:00,55.2,16.98,3079.10,564.697,396.533,10.0705,1.74,250.730,248.906,...,251.873,444.384,443.269,460.449,439.920,451.588,433.539,425.458,66.91,1.31
6,2017-03-10 01:00:00,55.2,16.98,3127.79,566.467,392.900,10.0713,1.74,250.313,252.202,...,253.477,446.185,444.571,452.306,431.328,443.548,444.575,431.251,66.91,1.31
7,2017-03-10 01:00:00,55.2,16.98,3152.93,558.777,397.002,10.0722,1.74,249.895,253.630,...,253.345,445.985,461.341,461.640,442.067,441.730,461.770,449.679,66.91,1.31
8,2017-03-10 01:00:00,55.2,16.98,3147.27,556.030,394.307,10.0730,1.74,250.137,251.104,...,250.884,446.686,478.385,459.103,455.074,439.798,457.738,455.915,66.91,1.31
9,2017-03-10 01:00:00,55.2,16.98,3142.58,565.857,393.105,10.0738,1.74,249.653,252.202,...,248.137,445.685,478.779,460.665,457.225,453.236,449.898,455.750,66.91,1.31


In [5]:
df['date'] = pd.to_datetime(df['date'])

In [6]:
print("Data Shape: ", df.shape)
print("Data Type: ", df.dtypes)

Data Shape:  (737453, 24)
Data Type:  date                            datetime64[us]
% Iron Feed                            float64
% Silica Feed                          float64
Starch Flow                            float64
Amina Flow                             float64
Ore Pulp Flow                          float64
Ore Pulp pH                            float64
Ore Pulp Density                       float64
Flotation Column 01 Air Flow           float64
Flotation Column 02 Air Flow           float64
Flotation Column 03 Air Flow           float64
Flotation Column 04 Air Flow           float64
Flotation Column 05 Air Flow           float64
Flotation Column 06 Air Flow           float64
Flotation Column 07 Air Flow           float64
Flotation Column 01 Level              float64
Flotation Column 02 Level              float64
Flotation Column 03 Level              float64
Flotation Column 04 Level              float64
Flotation Column 05 Level              float64
Flotation Column 06 Le

In [7]:
df.columns = (df.columns
              .str.lower()
              .str.replace('%', 'pct', regex=False)
              .str.replace(' ', '_')
              .str.strip('_'))
df.columns

Index(['date', 'pct_iron_feed', 'pct_silica_feed', 'starch_flow', 'amina_flow',
       'ore_pulp_flow', 'ore_pulp_ph', 'ore_pulp_density',
       'flotation_column_01_air_flow', 'flotation_column_02_air_flow',
       'flotation_column_03_air_flow', 'flotation_column_04_air_flow',
       'flotation_column_05_air_flow', 'flotation_column_06_air_flow',
       'flotation_column_07_air_flow', 'flotation_column_01_level',
       'flotation_column_02_level', 'flotation_column_03_level',
       'flotation_column_04_level', 'flotation_column_05_level',
       'flotation_column_06_level', 'flotation_column_07_level',
       'pct_iron_concentrate', 'pct_silica_concentrate'],
      dtype='str')

In [8]:
df.head(20)

,date,pct_iron_feed,pct_silica_feed,starch_flow,amina_flow,ore_pulp_flow,ore_pulp_ph,ore_pulp_density,flotation_column_01_air_flow,flotation_column_02_air_flow,...,flotation_column_07_air_flow,flotation_column_01_level,flotation_column_02_level,flotation_column_03_level,flotation_column_04_level,flotation_column_05_level,flotation_column_06_level,flotation_column_07_level,pct_iron_concentrate,pct_silica_concentrate
0,2017-03-10 01:00:00,55.2,16.98,3019.53,557.434,395.713,10.0664,1.74,249.214,253.235,...,250.884,457.396,432.962,424.954,443.558,502.255,446.370,523.344,66.91,1.31
1,2017-03-10 01:00:00,55.2,16.98,3024.41,563.965,397.383,10.0672,1.74,249.719,250.532,...,248.994,451.891,429.560,432.939,448.086,496.363,445.922,498.075,66.91,1.31
2,2017-03-10 01:00:00,55.2,16.98,3043.46,568.054,399.668,10.0680,1.74,249.741,247.874,...,248.071,451.240,468.927,434.610,449.688,484.411,447.826,458.567,66.91,1.31
3,2017-03-10 01:00:00,55.2,16.98,3047.36,568.665,397.939,10.0689,1.74,249.917,254.487,...,251.147,452.441,458.165,442.865,446.210,471.411,437.690,427.669,66.91,1.31
4,2017-03-10 01:00:00,55.2,16.98,3033.69,558.167,400.254,10.0697,1.74,250.203,252.136,...,248.928,452.441,452.900,450.523,453.670,462.598,443.682,425.679,66.91,1.31
5,2017-03-10 01:00:00,55.2,16.98,3079.10,564.697,396.533,10.0705,1.74,250.730,248.906,...,251.873,444.384,443.269,460.449,439.920,451.588,433.539,425.458,66.91,1.31
6,2017-03-10 01:00:00,55.2,16.98,3127.79,566.467,392.900,10.0713,1.74,250.313,252.202,...,253.477,446.185,444.571,452.306,431.328,443.548,444.575,431.251,66.91,1.31
7,2017-03-10 01:00:00,55.2,16.98,3152.93,558.777,397.002,10.0722,1.74,249.895,253.630,...,253.345,445.985,461.341,461.640,442.067,441.730,461.770,449.679,66.91,1.31
8,2017-03-10 01:00:00,55.2,16.98,3147.27,556.030,394.307,10.0730,1.74,250.137,251.104,...,250.884,446.686,478.385,459.103,455.074,439.798,457.738,455.915,66.91,1.31
9,2017-03-10 01:00:00,55.2,16.98,3142.58,565.857,393.105,10.0738,1.74,249.653,252.202,...,248.137,445.685,478.779,460.665,457.225,453.236,449.898,455.750,66.91,1.31


## Duplicate & quality checks
Check for duplicate rows, missing values, and out-of-range numbers, then drop exact duplicates.

In [9]:
df.isna().sum()

date                            0
pct_iron_feed                   0
pct_silica_feed                 0
starch_flow                     0
amina_flow                      0
ore_pulp_flow                   0
ore_pulp_ph                     0
ore_pulp_density                0
flotation_column_01_air_flow    0
flotation_column_02_air_flow    0
flotation_column_03_air_flow    0
flotation_column_04_air_flow    0
flotation_column_05_air_flow    0
flotation_column_06_air_flow    0
flotation_column_07_air_flow    0
flotation_column_01_level       0
flotation_column_02_level       0
flotation_column_03_level       0
flotation_column_04_level       0
flotation_column_05_level       0
flotation_column_06_level       0
flotation_column_07_level       0
pct_iron_concentrate            0
pct_silica_concentrate          0
dtype: int64

In [10]:
df.duplicated().value_counts()

False    736282
True       1171
Name: count, dtype: int64

In [11]:
dups = df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist())

In [12]:
dups["date"].value_counts()

date
2017-07-11 07:00:00    180
2017-07-11 08:00:00    180
2017-07-11 09:00:00    180
2017-07-13 15:00:00    180
2017-07-13 14:00:00    123
2017-06-28 16:00:00     85
2017-07-11 06:00:00     82
2017-07-11 10:00:00     47
2017-07-13 16:00:00     40
2017-07-26 11:00:00     38
2017-06-29 15:00:00     17
2017-06-28 15:00:00     15
2017-06-29 14:00:00     11
2017-06-01 15:00:00      6
2017-03-12 01:00:00      2
Name: count, dtype: int64

In [13]:
df = df.drop_duplicates().reset_index(drop=True)

In [14]:
for col in df.select_dtypes(include='number').columns:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    outliers = (~df[col].between(Q1 - 1.5*IQR, Q3 + 1.5*IQR)).sum()
    print(f"{col}: {outliers} outliers ({outliers/len(df)*100:.2f}%)")

pct_iron_feed: 0 outliers (0.00%)
pct_silica_feed: 0 outliers (0.00%)
starch_flow: 1090 outliers (0.15%)
amina_flow: 8592 outliers (1.17%)
ore_pulp_flow: 108401 outliers (14.72%)
ore_pulp_ph: 10986 outliers (1.49%)
ore_pulp_density: 43195 outliers (5.87%)
flotation_column_01_air_flow: 0 outliers (0.00%)
flotation_column_02_air_flow: 0 outliers (0.00%)
flotation_column_03_air_flow: 25 outliers (0.00%)
flotation_column_04_air_flow: 54099 outliers (7.35%)
flotation_column_05_air_flow: 29559 outliers (4.01%)
flotation_column_06_air_flow: 3161 outliers (0.43%)
flotation_column_07_air_flow: 194 outliers (0.03%)
flotation_column_01_level: 807 outliers (0.11%)
flotation_column_02_level: 1872 outliers (0.25%)
flotation_column_03_level: 353 outliers (0.05%)
flotation_column_04_level: 655 outliers (0.09%)
flotation_column_05_level: 2982 outliers (0.41%)
flotation_column_06_level: 2672 outliers (0.36%)
flotation_column_07_level: 795 outliers (0.11%)
pct_iron_concentrate: 1590 outliers (0.22%)
pct_

In [15]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,736282,2017-06-16 02:30:44.009768,2017-03-10 01:00:00,2017-05-04 21:00:00,2017-06-16 12:00:00,2017-07-29 09:00:00,2017-09-09 23:00:00,NaN
pct_iron_feed,736282.0,56.298307,42.74,52.67,56.08,59.72,65.78,5.160365
pct_silica_feed,736282.0,14.648984,1.31,8.94,13.85,19.6,33.4,6.810741
starch_flow,736282.0,2869.636615,0.002026,2075.07,3020.23,3728.93,6300.23,1216.017896
amina_flow,736282.0,488.165523,241.669,431.835848,504.3525,553.33575,739.538,91.254428
ore_pulp_flow,736282.0,397.570736,376.249,394.248,399.238,402.967,418.641,9.705444
ore_pulp_ph,736282.0,9.767315,8.75334,9.52705,9.79746,10.0378,10.8081,0.387176
ore_pulp_density,736282.0,1.680424,1.51982,1.64739,1.69758,1.72838,1.85325,0.069206
flotation_column_01_air_flow,736282.0,280.119813,175.51,250.278,299.341,300.147,373.871,29.633831
flotation_column_02_air_flow,736282.0,277.121249,175.156,250.448,296.202,300.686,375.992,30.157126


## Save clean data → MySQL
Write the cleaned dataset out and load it into the `plant_data` table.

In [16]:
df.to_csv("IronProcess_Plant_clean.csv", index=False)

In [20]:
load_dotenv()

False

In [21]:
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

In [23]:
df.to_sql('plant_data', engine, if_exists='append', index=False)

736282